# Minimal working project
This notebook walks through the basic parts of the main qualitative analysis loop we will use to programmatically call an AI model API to score free recall strings from participants in an EEG study. In this example, I use the free tier of google gemini to see how the typical workflow for prompting an AI API is set up. Considering a couple different models to use, balancing cost and their different analysis personalities. 

In [6]:
# import libraries
from dotenv import load_dotenv
import os
import time
import pandas as pd
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
from google.genai.errors import ServerError

# load in env variables
load_dotenv()

True

Define the desired output structure using Pydantic

In [3]:
class EvalResult(BaseModel):
    score: int = Field(description='An accuracy score from 1 to 5 based on the rubric.')
    justification: str = Field(description='A concise explanation of the score assigned to a response.')
    missing_facts: list[str] = Field(description='List of key facts missing from a response.')

Define baseline text, rubric, and some sample participant data that mirrors the structure of Mohsen's data

In [4]:
SCORING_RUBRIC = '''
Score the participant's free recall response against the Baseline Text for accuracy.
Assign an accuracy score from 1 to 5 using the criterion below:
5: Recalled all major details with perfect accuracy.
3: Recalled general premise of scene but missed several critical details.
1: Completely inaccurate or irrelevant recall.
'''

BASELINE_TEXT = "The Treaty of Versailles was signed on June 28, 1919, exactly five years after the assassination of Archduke Franz Ferdinand, ending World War I."

# Simulated pipeline dataset
dataset = [
    {"participant_id": "1", "recall_data": "The Treaty of Versailles ended W1 in June 1919. It was signed exactly 5 years after Franz Ferdinand was killed."},
    {"participant_id": "2", "recall_data": "A treaty was signed in 1919 that ended World War II after an archduke died."},
    {"participant_id": "3", "recall_data": "They signed a peace agreement in France somewhere around the early 1900s."}
]

Initialize gemini client

In [5]:
model = 'gemini-3.8-flash'
max_retries = 5
backoff_delay = 2

In [7]:
client = genai.Client()

results = []

print('Connected to client, starting pipeline...')

# loop over rows in the dataset
for row in dataset:
    pid = row['participant_id']
    recall = row['recall_data']
    
    # prompt reused for each data point
    prompt = f'''
    You are an expert qualitative data evaluator. Analyze the following participant free recall responses for accuracy against the provided baseline text and assign a score according to the constraints of the rubric.
    
    [Baseline Text]: {BASELINE_TEXT}
    [Participant Recall]: {recall}
    '''
    
    success = False
    
    # try more attempts if unsuccessful
    for attempt in range(max_retries):
        # reach out to API for response
        try:
            # request evaluation from 2.5 Flash for speed
            response = client.models.generate_content(
                model=model, 
                contents=prompt, 
                config=types.GenerateContentConfig(
                    system_instruction=SCORING_RUBRIC,
                    response_mime_type='application/json', # enforce JSON format
                    response_schema=EvalResult,            # use our schema
                    temperature=0.1                        # more reliability with lower temp apparently
                )
            )
            
            # if it works, break out of retry loop
            success = True
            break
        
        except ServerError as e:
            # check if it is a 503
            if e.code == 503:
                print(f'Attempt {attempt+1}/{max_retries} hit 503, retrying in {backoff_delay}s...')
                time.sleep(backoff_delay)
                backoff_delay *= 2 # double wait time for subsequent delays
            else:
                # if not 503, raise it back up
                raise e
            
        except Exception as e:
            print(f'Non-server error encountered on {pid}: {e}')
    
    if success:
        try:
            # parse JSON string and store it as pd.df
            import json
            structured_data = json.loads(response.text)
            
            record = {
                'participant_id': pid, 
                'recall_data': recall, 
                'assigned_score': structured_data.get('score'),
                'justification': structured_data.get('justification'),
                'missing_facts': ', '.join(structured_data.get('missing_facts', []))
            }
            results.append(record)

            print(f'Successfully scored {pid}: Score = {record['assigned_score']}')

        except Exception as parse_error:
            print(f'Failed to parse JSON schema for {pid}: {parse_error}')
    else:
        print(f'Failed to process {pid} after {max_retries} attempts!')
            
# view and stash the results
if not os.path.exists('../model_output'):
    os.makedirs('../model_output')

df = pd.DataFrame(results)
df.to_csv(os.path.join('..\\model_output', f'{model}-results.csv'), index=False)
print(f'\nAnalysis complete, stashing results in model_results/{model}-results.csv')


Connected to client, starting pipeline...
Successfully scored 1: Score = 4
Successfully scored 2: Score = 2
Successfully scored 3: Score = 2

Analysis complete, stashing results in model_results/gemini-3.8-flash-results.csv
